In [ ]:
# create_trainset.ipynb
#
# Generates a synthetic training dataset for a neural-network LBM collision operator.
#
# Pipeline overview
# -----------------
# 1. Sample random macroscopic fluid states (density rho, velocity u).
# 2. Compute the equilibrium population f_eq from those states.
# 3. Add a random non-equilibrium perturbation f_neq (= f_pre - f_eq).
# 4. Apply the BGK collision rule to get f_post (the "label").
# 5. Keep only samples with strictly positive populations (negative values are unphysical).
# 6. Save (f_pre, f_post, f_eq) tuples for training.

import numpy as np
import matplotlib.pyplot as plt

from utils import *

In [ ]:
def compute_rho_u(num_samples, rho_min=0.95, rho_max=1.05, u_abs_min=0.0, u_abs_max=0.01):
    """Sample random macroscopic fluid states.

    In LBM the macroscopic (observable) quantities are moments of the population f:
        rho      = Σ_i f_i           — fluid density (zeroth moment)
        rho * u  = Σ_i f_i * c_i    — momentum (first moment); c_i are lattice velocities

    We draw rho and |u| uniformly from physical ranges, then pick a random
    flow direction theta to get the (ux, uy) velocity components.

    Returns
    -------
    rho : (num_samples,)    — density per sample
    u   : (num_samples, 2)  — velocity vector [ux, uy] per sample
    """
    rho   = np.random.uniform(rho_min, rho_max, size=num_samples)   # density: near 1 in lattice units
    u_abs = np.random.uniform(u_abs_min, u_abs_max, size=num_samples) # speed: kept small (low Mach)
    theta = np.random.uniform(0, 2*np.pi, size=num_samples)           # random flow direction

    ux = u_abs * np.cos(theta)  # x-component of velocity
    uy = u_abs * np.sin(theta)  # y-component of velocity
    u  = np.array([ux, uy]).transpose()  # shape (num_samples, 2)

    return rho, u


def compute_f_rand(num_samples, sigma_min, sigma_max):
    """Generate random non-equilibrium perturbations f_neq that conserve mass and momentum.

    The full population is split as  f = f_eq + f_neq.
    f_neq encodes deviations from local equilibrium (e.g. velocity gradients, stress).
    It must satisfy the conservation constraints so it doesn't accidentally shift
    the macroscopic state (rho, u) away from the values used to compute f_eq:
        Σ_i f_neq_i        = 0   (no extra mass)
        Σ_i f_neq_i c_{ix} = 0   (no extra x-momentum)
        Σ_i f_neq_i c_{iy} = 0   (no extra y-momentum)

    Strategy: draw Gaussian noise, then subtract its mass and momentum projections.

    Parameters
    ----------
    sigma_min/max : range for the noise amplitude; controls how far f is from equilibrium.

    Returns
    -------
    f_rand : (num_samples, 9) — the conservative non-equilibrium perturbation
    """
    Q  = 9      # number of lattice directions (D2Q9)

    # Projection coefficients derived from the D2Q9 stencil weights.
    # K0 removes the net mass contribution; K1 removes net momentum.
    K0 = 1/9.   # = 1/Q (uniform mass projection)
    K1 = 1/6.   # = 1/(Σ c_{ix}^2) = 1/(Σ c_{iy}^2) for D2Q9

    f_rand = np.zeros((num_samples, Q))

    # Draw a single sigma per sample (or a fixed one when sigma_min == sigma_max)
    if sigma_min == sigma_max:
        sigma = sigma_min * np.ones(num_samples)
    else:
        sigma = np.random.uniform(sigma_min, sigma_max, size=num_samples)

    for i in range(num_samples):
        # Raw Gaussian noise — not yet conservative
        f_rand[i, :] = np.random.normal(0, sigma[i], size=(1, Q))

        # Compute the spurious mass and momentum the noise would introduce
        rho_hat = np.sum(f_rand[i, :])          # would-be density change
        ux_hat  = np.sum(f_rand[i, :] * c[:, 0]) # would-be x-momentum change
        uy_hat  = np.sum(f_rand[i, :] * c[:, 1]) # would-be y-momentum change

        # Project out (subtract) the spurious contributions so constraints hold exactly
        f_rand[i, :] = f_rand[i, :] - K0*rho_hat - K1*ux_hat*c[:, 0] - K1*uy_hat*c[:, 1]

    return f_rand


def compute_f_pre_f_post(f_eq, f_neq, tau_min=1, tau_max=1):
    """Apply the BGK collision operator.

    BGK (Bhatnagar-Gross-Krook) is the simplest collision rule:
        f_post = f_pre + (1/tau) * (f_eq - f_pre)

    This is a linear relaxation towards equilibrium with timescale tau.
    tau is linked to the kinematic viscosity by:  nu = cs² * (tau - 0.5)
    so larger tau → more viscous fluid, slower relaxation.

    Parameters
    ----------
    f_eq  : (N, 9) — equilibrium populations (the "target" state)
    f_neq : (N, 9) — non-equilibrium perturbation
    tau_min/max    — range for the relaxation time tau (sampled uniformly per sample)

    Returns
    -------
    tau    : (N,)   — relaxation time used for each sample
    f_pre  : (N, 9) — pre-collision populations  (= f_eq + f_neq, network input)
    f_post : (N, 9) — post-collision populations (= BGK result,   network label)
    """
    tau = np.random.uniform(tau_min, tau_max, size=f_eq.shape[0])

    f_pre  = f_eq + f_neq                              # pre-collision state
    f_post = f_pre + (1/tau[:, None]) * (f_eq - f_pre) # BGK relaxation step

    return tau, f_pre, f_post


def delete_negative_samples(n_samples, f_eq, f_pre, f_post):
    """Remove samples that contain any negative population value.

    Populations f_i represent (proportional) particle densities; they must be
    non-negative everywhere.  Negative values arise when the random f_neq
    perturbation is too large relative to f_eq — such samples are unphysical
    and would corrupt training.  We discard the entire sample if any of the
    three arrays (f_eq, f_pre, f_post) has a negative entry.
    """
    # Find row indices where at least one population is negative
    i_neg_f_eq   = np.where(np.sum(f_eq   < 0, axis=1) > 0)[0]
    i_neg_f_pre  = np.where(np.sum(f_pre  < 0, axis=1) > 0)[0]
    i_neg_f_post = np.where(np.sum(f_post < 0, axis=1) > 0)[0]

    # Union of all bad indices
    i_neg_f = np.concatenate((i_neg_f_pre, i_neg_f_post, i_neg_f_eq))

    # Drop those rows from all three arrays consistently
    f_eq   = np.delete(np.copy(f_eq),   i_neg_f, axis=0)
    f_pre  = np.delete(np.copy(f_pre),  i_neg_f, axis=0)
    f_post = np.delete(np.copy(f_post), i_neg_f, axis=0)

    return f_eq, f_pre, f_post

In [ ]:
# -----------------------------------------------------------------------
# Hyperparameters
# -----------------------------------------------------------------------

n_samples = 100_000  # total number of (f_pre, f_post) training pairs to collect

# Velocity range: u_abs_max = 0.01 keeps the Mach number << 1 (low-Mach regime
# where the LBM approximation of the Navier-Stokes equations is valid)
u_abs_min = 1e-15   # effectively zero — allows near-rest states
u_abs_max = 0.01

# Non-equilibrium amplitude range: sigma controls how far f_pre deviates from f_eq.
# Larger sigma → stronger non-equilibrium effects (higher strain rates / stress).
sigma_min = 1e-15   # effectively zero — allows near-equilibrium states
sigma_max = 5e-4

# -----------------------------------------------------------------------
# D2Q9 stencil
# -----------------------------------------------------------------------
# c   : (9, 2) lattice velocity vectors
# w   : (9,)   quadrature weights
# cs2 : lattice speed of sound squared (= 1/3)
# compute_feq : JIT-compiled function that evaluates the equilibrium distribution
Q = 9
c, w, cs2, compute_feq = LB_stencil()

# -----------------------------------------------------------------------
# Pre-allocate output arrays
# -----------------------------------------------------------------------
# Each row is one sample; each column is one of the 9 lattice directions.
fPreLst  = np.empty((n_samples, Q))  # pre-collision populations  (network input)
fPostLst = np.empty((n_samples, Q))  # post-collision populations (network label)
fEqLst   = np.empty((n_samples, Q))  # equilibrium populations    (used in AlgReconstruction)

# -----------------------------------------------------------------------
# Data generation loop
# -----------------------------------------------------------------------
# We generate batches of n_samples and discard physically invalid ones
# (negative populations).  We repeat until the output arrays are full.

idx = 0  # number of valid samples collected so far

while idx < n_samples:

    # Step 1 — sample random macroscopic states (rho, u) for a full batch
    rho, u = compute_rho_u(n_samples)

    # Reshape for compute_feq which expects (N, 1) columns
    rho = rho[:, np.newaxis]      # fluid density,      shape (N, 1)
    ux  = u[:, 0][:, np.newaxis]  # x-velocity,         shape (N, 1)
    uy  = u[:, 1][:, np.newaxis]  # y-velocity,         shape (N, 1)

    # Step 2 — compute the equilibrium distribution f_eq from the macroscopic state.
    # compute_feq fills a (N, 1, 9) array; we squeeze the spatial dimension to (N, 9).
    f_eq = np.zeros((n_samples, 1, Q))
    f_eq = compute_feq(f_eq, rho, ux, uy, c, w)[:, 0, :]  # shape (N, 9)

    # Step 3 — add a random conservative non-equilibrium part.
    # f_neq satisfies Σ f_neq = 0 and Σ f_neq * c_i = 0, so it does not shift
    # the macroscopic state (rho, u) that was used to compute f_eq.
    f_neq = compute_f_rand(n_samples, sigma_min, sigma_max)  # shape (N, 9)

    # Step 4 — apply the BGK collision:  f_post = f_pre + (1/tau)*(f_eq - f_pre)
    # tau (the relaxation time) is drawn uniformly in [tau_min, tau_max].
    tau, f_pre, f_post = compute_f_pre_f_post(f_eq, f_neq)

    # Step 5 — discard any sample that has a negative population in f_eq, f_pre, or f_post.
    f_eq, f_pre, f_post = delete_negative_samples(n_samples, f_eq, f_pre, f_post)

    # Step 6 — append valid samples into the output arrays, stopping at n_samples.
    non_negatives = f_pre.shape[0]           # how many survived the filter
    idx1        = min(idx + non_negatives, n_samples)
    to_be_added = min(n_samples - idx, non_negatives)

    fPreLst[ idx:idx1] = f_pre[ :to_be_added]
    fPostLst[idx:idx1] = f_post[:to_be_added]
    fEqLst[  idx:idx1] = f_eq[  :to_be_added]

    idx += non_negatives  # advance counter (may overshoot; loop exits when idx >= n_samples)

In [ ]:
# Save the dataset.
#
# Stored arrays:
#   f_pre  : (n_samples, 9) — pre-collision populations   → network input
#   f_post : (n_samples, 9) — post-collision populations  → network target (label)
#   f_eq   : (n_samples, 9) — equilibrium populations     → used by AlgReconstruction
#                             to enforce conservation laws at inference time

np.savez('example_dataset.npz',
         f_pre  = fPreLst,
         f_post = fPostLst,
         f_eq   = fEqLst,
        )